# Lab 08 — Memory for Agents

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

**Learning objectives** — after this lab you can:

- state the course's **terminology contract** — the context window is *working memory*; the word *memory* is reserved for state that survives the run — and the **three structural reasons** the window fails as memory (volatility, cost, limits),
- classify agent state with the **CoALA taxonomy** (short-term / episodic / semantic / procedural) and place any mechanism on the four-step **memory lifecycle** (experience → write → retrieve → inject, with forgetting as the loop),
- build a **token meter** and reproduce the lecture's "re-sending is re-paying" cost argument,
- implement and compare **truncation with pinning** and **engineered compaction** (80% trigger, explicit keep/drop lists) — and *measure* what each one loses,
- give the research agent the lecture's **14-line memory-file pattern** (`memory_read` / `memory_append`) with a write policy and an end-of-run **consolidation pass**,
- run the lecture's lab assignment: **two sessions on related topics** (Monday: battery recycling → Wednesday: lithium supply chains), with and without memory, and **measure — not guess — what carries over**, and what carries over that should not have,
- implement **forgetting** (TTL expiry + supersession) as a separate consolidation pass (report task R2).

> ⏱️ Estimated time: 90–120 minutes. The four live agent runs in Part F take a few minutes
> each on a local model — start them early and read on while they run.

## Theory recap — why the context window is not memory

### Working memory, not memory

The LLM is **stateless**: every API call is a pure function — tokens in, tokens out, nothing
kept. A chatbot appears to remember only because the runtime **replays the full message list
on every call**. The right analogy from cognitive psychology is **working memory**: small,
fast, holding exactly the current task — and gone the moment the task ends. The course's
terminology contract: *we reserve the word memory for state that survives the run.* Nothing
in the context window ever does. Three structural failures follow (design properties, not
bugs): **volatility** — the message list is process state, garbage-collected at exit;
**cost** — replay means every kept token is *re-billed on every call*: a 100k-token context
over a 40-step run bills about **4 million input tokens** (prompt caching discounts the
replay, but entries expire in minutes and only cover an unchanged prefix — the principle
stands); **limits** — windows are finite, and the soft limit bites first: models attend
poorly to the middle of long contexts (*lost in the middle*, Liu et al., 2024). A full
window is not a usable window.

### A taxonomy and a lifecycle

The CoALA framing (Sumers et al., 2024, after Tulving, 1972) sorts agent memory into four
types. **Short-term (working)**: the window — this run's messages, tool results, scratchpad.
Three long-term types live outside it: **episodic** — records of past runs (*what
happened?*: last report's sources and dead ends); **semantic** — facts and preferences
detached from any episode (the user wants APA citations and EU sources); **procedural** —
learned how-to (standing rules like *always verify quotes before citing*; plus the weights
themselves, which we cannot touch). Each type wants its own store — log, key-value/vector
store, memory file — with its own write path, read path and lifetime; an undifferentiated
memory blob buys the worst of all four worlds. Every long-term system implements the same
lifecycle: **experience → write → retrieve → inject**, with **forgetting** pruning the store
between runs. Injection closes the loop with section one: a memory is only useful at the
moment it re-enters the window.

### Managing the window

Three families, from crude to deliberate. **Truncation**: drop the oldest messages, keep a
sliding window; trivial and free, with one mandatory refinement — the system prompt and task
goal are *pinned*. Its failure mode is **silence**: an early user constraint falls off the
cliff and nothing signals the loss. **Compaction**: replace old messages with a
model-written summary — trigger at ~**80%** of the budget (compaction itself needs
headroom), *keep* decisions, constraints, source verdicts, open questions, file names;
*drop* raw tool dumps and re-fetchable page text. Every compaction is a lossy, irreversible
bet on what the future run will need (MemGPT framed this as OS-style paging, Packer et al.,
2023; Claude Code ships it as auto-compact). **Memory files**: a persistent `notes.md` read
at session start and edited as the agent works — human-readable, versionable, portable; the
CLAUDE.md pattern. Key property: **notes survive compaction and restarts** — the two
techniques compose.

### Policies and the architecture fork

**Write**: store what is *stable, reusable, expensive to re-derive*; distill first (the
conclusion, not the transcript); deduplicate — a useless memory pollutes every future
retrieval. **Read**: session-start core, on-demand tool queries, similarity-triggered
retrieval (next week's machinery); rank by *recency × importance × relevance* (Park et al.,
2023); at the read boundary **precision beats recall** — an irrelevant memory actively
misleads. **Forget**: TTL expiry, usage-based decay, supersession, explicit deletion (also a
legal duty — GDPR Art. 17). Architecturally: **memory as a tool** is deliberate, auditable,
token-frugal — but the model may simply not call it; **automatic injection** is reliable but
silent, costly, and the surface memory poisoning exploits. Shipping products converge on the
**hybrid**: inject a small curated core at start, tools for the long tail.

### This lab

Exactly what the lecture announced: *add the memory-file tools to your research agent, run
it twice on related topics, and measure what carries over* — and what carries over that
should not have. We first build the working-memory plumbing (token meter, truncation,
compaction), then the memory file with write policy and consolidation, then the two-session
experiment: **Monday** battery recycling (with the user's standing requirements), **Wednesday**
lithium supply chains (requirements *not* restated). Search runs against a tiny offline
corpus in `data/` — one page of which is trying to poison your memory (R3).

> **Q (not exam-relevant):** Where does the episodic/semantic distinction originally come from, and why does the lecture cite the source?
<details><summary>Click for answer</summary>

Tulving (1972) introduced it in cognitive psychology: episodic memory is remembering
experienced events; semantic memory is knowing facts without remembering their acquisition.
The agent literature borrowed the vocabulary (via CoALA, Sumers et al., 2024); citing the
original keeps the borrowed analogy honest and signals that the terms have precise meanings.
</details>

## Part A — Setup & Ollama connectivity

One tool-capable local model is enough (default `qwen2.5:7b`, override via the environment
variable `OLLAMA_MODEL`). Every LLM cell degrades gracefully if Ollama is not reachable —
the token meter, truncation, the memory-file mechanics and the forgetting pass (R2) all run
**without** a model; only compaction and the live Part F experiment need one.

In [ ]:
import os
import re
import json
import time
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")   # any tool-capable 7-30B model works
OLLAMA_OK = False
AVAILABLE = []


def _field(obj, name, default=None):
    """Read a field from dict-like or attribute-style Ollama responses (client versions differ)."""
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


try:
    import ollama
    listing = ollama.list()
    for m in _field(listing, "models", []) or []:
        name = _field(m, "model", None) or _field(m, "name", "")
        if name:
            AVAILABLE.append(name)
    OLLAMA_OK = True
    print("Ollama is running. Local models:", ", ".join(AVAILABLE) or "(none)")
    base = MODEL.split(":")[0]
    if not any(a == MODEL or a.split(":")[0] == base for a in AVAILABLE):
        print(f"Model {MODEL!r} not found — run `ollama pull {MODEL}`.")
except Exception as exc:
    print("Could not reach Ollama:", exc)
    print("→ Start Ollama with `ollama serve` and pull the model with `ollama pull qwen2.5:7b`.")
    print("  All window-management mechanics below still run; only the live runs are skipped.")

> **Q:** Explain why an LLM-based chatbot appears to remember a conversation even though the model is stateless.
<details><summary>Click for answer</summary>

The model retains nothing between API calls: each call is a pure function from input tokens
to output tokens. The surrounding runtime keeps the message list and replays the entire
history on every call, so the model re-reads the whole conversation each time. The apparent
memory is a replay performed by orchestration code, not state inside the model.
</details>

## Part B — The research agent, its corpus, and a token meter

The running example stays the course's **research agent**: brief → search → evaluate sources
→ draft a Markdown report. Search runs over an **offline corpus of eight saved pages** in
`data/corpus.json` (synthetic; two related topics — battery recycling and lithium supply
chains — with source types from EU reports to a marketing blog, so the user's source
preferences actually bite).

The second cell builds the instrument this lab lives by: a **token meter**. We approximate
tokens as characters/4 — crude, but a *measured* approximation beats a guessed one, and it
lets us reproduce the lecture's bigfact: **re-sending is re-paying**.

In [ ]:
CORPUS_PATH = "data/corpus.json"
corpus = json.loads(Path(CORPUS_PATH).read_text(encoding="utf-8"))
ALL_IDS = {d["id"] for d in corpus}
print(f"{len(corpus)} saved pages in the offline corpus:", ", ".join(sorted(ALL_IDS)))


def score(doc, terms):
    """Keyword relevance of one document: hits in the title count double."""
    title, text = doc["title"].lower(), doc["text"].lower()
    return sum(2 * (t in title) + (t in text) for t in terms)


def web_search(query: str) -> str:
    """Search the saved research corpus; return the three MOST relevant sources."""
    terms = [t for t in query.lower().split() if len(t) > 2]
    ranked = sorted(corpus, key=lambda d: score(d, terms), reverse=True)
    hits = [d for d in ranked if score(d, terms) > ___][:3]
    if not hits:
        return "No results for this query. Try different terms."
    return "\n\n".join(f"[{d['id']}] {d['title']} ({d['source_type']}, {d['date']}, "
                       f"region: {d['region']})\nURL: {d['url']}\n{d['text']}" for d in hits)


def save_report(filename: str, content: str) -> str:
    """Save a Markdown report to the reports/ folder; returns a confirmation."""
    outdir = Path("reports")
    outdir.mkdir(exist_ok=True)
    out = outdir / Path(filename).name
    out.write_text(content, encoding="utf-8")
    return f"Report saved to {out} ({len(content)} characters)."


TOOLS = {"web_search": web_search, "save_report": ___}   # the registry: a plain dict

SCHEMAS = [
    {"type": "function", "function": {
        "name": "web_search",
        "description": ("Search the saved research corpus. Returns the three most relevant "
                        "sources with id, title, source type, region, URL and text."),
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string",
                                                "description": "search terms"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "save_report",
        "description": "Save the final Markdown report to disk.",
        "parameters": {"type": "object",
                       "properties": {"filename": {"type": "string"},
                                      "content": {"type": "string",
                                                  "description": "Markdown text"}},
                       "required": ["filename", "content"]}}},
]


def to_dict(msg):
    """Normalise an Ollama message (object or dict) into a plain, JSON-serialisable dict."""
    if hasattr(msg, "model_dump"):
        msg = msg.model_dump(exclude_none=True)
    keep = {k: msg[k] for k in ("role", "content", "tool_calls") if msg.get(k) is not None}
    keep.setdefault("content", "")
    return json.loads(json.dumps(keep, default=str))


def run_tool(call, registry=None):
    """Execute ONE tool call; wrap the observation as a role='tool' message (S03 convention:
    tool errors are observations, not crashes)."""
    registry = registry if registry is not None else TOOLS
    fn = call["function"]["name"]
    args = call["function"]["arguments"]
    if isinstance(args, str):
        args = json.loads(args)
    try:
        result = registry[fn](**___)
    except Exception as exc:
        result = f"Tool error: {exc}"
    return {"role": "tool", "tool_name": fn, "content": str(result)}


print("\n--- smoke test ---")
print(web_search("battery recycling europe")[:320], "…")

<details>
<summary><b>Click here for the solution</b></summary>

```python
CORPUS_PATH = "data/corpus.json"
corpus = json.loads(Path(CORPUS_PATH).read_text(encoding="utf-8"))
ALL_IDS = {d["id"] for d in corpus}
print(f"{len(corpus)} saved pages in the offline corpus:", ", ".join(sorted(ALL_IDS)))


def score(doc, terms):
    """Keyword relevance of one document: hits in the title count double."""
    title, text = doc["title"].lower(), doc["text"].lower()
    return sum(2 * (t in title) + (t in text) for t in terms)


def web_search(query: str) -> str:
    """Search the saved research corpus; return the three MOST relevant sources."""
    terms = [t for t in query.lower().split() if len(t) > 2]
    ranked = sorted(corpus, key=lambda d: score(d, terms), reverse=True)
    hits = [d for d in ranked if score(d, terms) > 0][:3]
    if not hits:
        return "No results for this query. Try different terms."
    return "\n\n".join(f"[{d['id']}] {d['title']} ({d['source_type']}, {d['date']}, "
                       f"region: {d['region']})\nURL: {d['url']}\n{d['text']}" for d in hits)


def save_report(filename: str, content: str) -> str:
    """Save a Markdown report to the reports/ folder; returns a confirmation."""
    outdir = Path("reports")
    outdir.mkdir(exist_ok=True)
    out = outdir / Path(filename).name
    out.write_text(content, encoding="utf-8")
    return f"Report saved to {out} ({len(content)} characters)."


TOOLS = {"web_search": web_search, "save_report": save_report}   # the registry: a plain dict

SCHEMAS = [
    {"type": "function", "function": {
        "name": "web_search",
        "description": ("Search the saved research corpus. Returns the three most relevant "
                        "sources with id, title, source type, region, URL and text."),
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string",
                                                "description": "search terms"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "save_report",
        "description": "Save the final Markdown report to disk.",
        "parameters": {"type": "object",
                       "properties": {"filename": {"type": "string"},
                                      "content": {"type": "string",
                                                  "description": "Markdown text"}},
                       "required": ["filename", "content"]}}},
]


def to_dict(msg):
    """Normalise an Ollama message (object or dict) into a plain, JSON-serialisable dict."""
    if hasattr(msg, "model_dump"):
        msg = msg.model_dump(exclude_none=True)
    keep = {k: msg[k] for k in ("role", "content", "tool_calls") if msg.get(k) is not None}
    keep.setdefault("content", "")
    return json.loads(json.dumps(keep, default=str))


def run_tool(call, registry=None):
    """Execute ONE tool call; wrap the observation as a role='tool' message (S03 convention:
    tool errors are observations, not crashes)."""
    registry = registry if registry is not None else TOOLS
    fn = call["function"]["name"]
    args = call["function"]["arguments"]
    if isinstance(args, str):
        args = json.loads(args)
    try:
        result = registry[fn](**args)
    except Exception as exc:
        result = f"Tool error: {exc}"
    return {"role": "tool", "tool_name": fn, "content": str(result)}


print("\n--- smoke test ---")
print(web_search("battery recycling europe")[:320], "…")
```

</details>

In [ ]:
def approx_tokens(text: str) -> int:
    """Cheap token estimate: ~4 characters per token for English text."""
    return max(1, len(text) // ___)


def context_tokens(messages) -> int:
    """Approximate size of a message list as the model receives it on ONE call."""
    return sum(___(json.dumps(m, default=str)) for m in messages)


# The lecture's bigfact, reproduced: 100k tokens of context, a 40-step run.
per_call, steps = 100_000, 40
print(f"Billed input tokens for the run: {per_call * steps:,}")     # ≈ 4 million

# Growth model of our research agent: each step appends ~700 tokens of tool output
# on top of a 1,200-token start (system prompt + task).
grow = 1_200 + np.cumsum(np.full(40, 700))       # context size at each step
billed_keep = np.cumsum(___)                    # keep everything: the replay grows
billed_capped = np.cumsum(np.minimum(grow, 8_000))   # a window managed down to 8k

plt.figure(figsize=(7, 4))
plt.plot(billed_keep, label="keep everything")
plt.plot(billed_capped, label="window managed at 8k tokens")
plt.xlabel("agent step")
plt.ylabel("cumulative billed input tokens")
plt.title("Re-sending is re-paying")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print(f"After 40 steps: {billed_keep[-1]:,} (keep everything) vs "
      f"{billed_capped[-1]:,} (managed) billed input tokens")

<details>
<summary><b>Click here for the solution</b></summary>

```python
def approx_tokens(text: str) -> int:
    """Cheap token estimate: ~4 characters per token for English text."""
    return max(1, len(text) // 4)


def context_tokens(messages) -> int:
    """Approximate size of a message list as the model receives it on ONE call."""
    return sum(approx_tokens(json.dumps(m, default=str)) for m in messages)


# The lecture's bigfact, reproduced: 100k tokens of context, a 40-step run.
per_call, steps = 100_000, 40
print(f"Billed input tokens for the run: {per_call * steps:,}")     # ≈ 4 million

# Growth model of our research agent: each step appends ~700 tokens of tool output
# on top of a 1,200-token start (system prompt + task).
grow = 1_200 + np.cumsum(np.full(40, 700))       # context size at each step
billed_keep = np.cumsum(grow)                    # keep everything: the replay grows
billed_capped = np.cumsum(np.minimum(grow, 8_000))   # a window managed down to 8k

plt.figure(figsize=(7, 4))
plt.plot(billed_keep, label="keep everything")
plt.plot(billed_capped, label="window managed at 8k tokens")
plt.xlabel("agent step")
plt.ylabel("cumulative billed input tokens")
plt.title("Re-sending is re-paying")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print(f"After 40 steps: {billed_keep[-1]:,} (keep everything) vs "
      f"{billed_capped[-1]:,} (managed) billed input tokens")
```

</details>

> **Q:** A 100k-token context is replayed over a 40-step run. Roughly how many input tokens are billed — and what does prompt caching change about this?
<details><summary>Click for answer</summary>

Roughly 4 million input tokens (40 calls × 100k). Prompt caching reduces the price of the
unchanged prefix — often to around a tenth of the normal rate — but it does not eliminate
the replay: cache entries expire within minutes, only apply to an unchanged prefix, and
provide no persistence between sessions. The structural argument for a second storage tier
stands: working memory is paid for on every call.
</details>

## Part C — Truncation: the sliding window, and its silent failure

First technique, exactly as in the lecture: keep a **sliding window**, drop the oldest
messages, and **pin** the system prompt and the initial task so the agent never forgets what
it is doing mid-run. We test it on a **synthetic long-run transcript** (no LLM needed) in
which the user adds a constraint *mid-run* — "use ONLY European sources" — precisely the
message the lecture says falls off the cliff. `mentions()` is our loss detector: measured,
not guessed.

In [ ]:
def make_transcript(n_pages=10):
    """A synthetic research-run transcript with big tool dumps. The user adds a
    constraint MID-RUN (message index 3) — the one truncation will silently lose."""
    msgs = [{"role": "system", "content": "You are a careful research agent."},
            {"role": "user", "content": "Write a report on battery recycling in Europe "
                                        "and save it as report.md."},
            {"role": "assistant", "content": "Plan: search broadly, then evaluate sources."},
            {"role": "user", "content": "One more requirement: use ONLY European sources."}]
    for k in range(n_pages):
        doc = corpus[k % len(corpus)]
        msgs.append({"role": "assistant", "content": f"Fetching source {k + 1}.",
                     "tool_calls": [{"function": {"name": "web_search",
                                                  "arguments": {"query": doc["title"]}}}]})
        msgs.append({"role": "tool", "tool_name": "web_search",
                     "content": f"[{doc['id']}] {doc['title']}\n" + doc["text"] * 3})
    return msgs


def mentions(messages, needle):
    """Loss detector: does ANY message still mention the needle?"""
    return any(needle.lower() in str(m.get(___, "")).lower() for m in messages)


def truncate(messages, budget):
    """Sliding-window truncation with PINNING: the system prompt and the initial task
    (the first two messages) are never dropped; the OLDEST unpinned messages fall first."""
    pinned, rest = list(messages[:___]), list(messages[2:])
    while rest and context_tokens(pinned + rest) > budget:
        rest.pop(___)                                   # drop the oldest unpinned message
    return pinned + rest


transcript = make_transcript()
print(f"transcript: {len(transcript)} messages, ~{context_tokens(transcript):,} tokens")

BUDGET = 2_400          # deliberately tiny, so the mechanisms actually fire in this lab
short = truncate(transcript, BUDGET)
print(f"truncated : {len(short)} messages, ~{context_tokens(short):,} tokens")

for label, msgs in [("before", transcript), ("after ", short)]:
    print(f"{label} truncation — constraint 'ONLY European' present:",
          mentions(msgs, "only european"))

<details>
<summary><b>Click here for the solution</b></summary>

```python
def make_transcript(n_pages=10):
    """A synthetic research-run transcript with big tool dumps. The user adds a
    constraint MID-RUN (message index 3) — the one truncation will silently lose."""
    msgs = [{"role": "system", "content": "You are a careful research agent."},
            {"role": "user", "content": "Write a report on battery recycling in Europe "
                                        "and save it as report.md."},
            {"role": "assistant", "content": "Plan: search broadly, then evaluate sources."},
            {"role": "user", "content": "One more requirement: use ONLY European sources."}]
    for k in range(n_pages):
        doc = corpus[k % len(corpus)]
        msgs.append({"role": "assistant", "content": f"Fetching source {k + 1}.",
                     "tool_calls": [{"function": {"name": "web_search",
                                                  "arguments": {"query": doc["title"]}}}]})
        msgs.append({"role": "tool", "tool_name": "web_search",
                     "content": f"[{doc['id']}] {doc['title']}\n" + doc["text"] * 3})
    return msgs


def mentions(messages, needle):
    """Loss detector: does ANY message still mention the needle?"""
    return any(needle.lower() in str(m.get("content", "")).lower() for m in messages)


def truncate(messages, budget):
    """Sliding-window truncation with PINNING: the system prompt and the initial task
    (the first two messages) are never dropped; the OLDEST unpinned messages fall first."""
    pinned, rest = list(messages[:2]), list(messages[2:])
    while rest and context_tokens(pinned + rest) > budget:
        rest.pop(0)                                   # drop the oldest unpinned message
    return pinned + rest


transcript = make_transcript()
print(f"transcript: {len(transcript)} messages, ~{context_tokens(transcript):,} tokens")

BUDGET = 2_400          # deliberately tiny, so the mechanisms actually fire in this lab
short = truncate(transcript, BUDGET)
print(f"truncated : {len(short)} messages, ~{context_tokens(short):,} tokens")

for label, msgs in [("before", transcript), ("after ", short)]:
    print(f"{label} truncation — constraint 'ONLY European' present:",
          mentions(msgs, "only european"))
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- Pinning protects the *first two* messages only — that is exactly the lecture's mandatory
  refinement (system prompt + task goal), and exactly why the *mid-run* constraint at index 3
  is unprotected: it is ordinary conversation history and the oldest of it falls first.
- `mentions()` turns "the agent forgot" from an anecdote into a boolean you can assert on —
  the lab's measure-not-guess discipline in one line.
- Note what did **not** happen: no error, no warning. The run continues with a
  plausible-looking context. That is the silent failure mode the lecture warned about.

</details>

> **Q:** Why is the failure mode of truncation described as *silent*, and why is that worse than a hard error?
<details><summary>Click for answer</summary>

Nothing signals that dropped content is gone: the agent continues confidently without the
constraint stated in an early message and behaves as if it never existed. A hard error halts
the run and gets fixed; silent constraint loss produces plausible-looking but wrong behavior
— the agent starts citing the wrong sources — that may only be noticed by the user, after
the damage.
</details>

## Part D — Compaction: engineered lossy compression

Second technique: replace the oldest messages with a **model-written summary**. Engineered
per the lecture, not a checkbox: trigger at **80% of the budget** (compaction needs
headroom), an explicit **keep list** (decisions, constraints, source verdicts, open
questions, file names) and **drop list** (raw page text, tool dumps — anything re-fetchable
in one call). The newest `KEEP_RECENT` messages survive verbatim so the agent keeps its
immediate train of thought.

Then we run the same measurement as in Part C: does the mid-run constraint survive the
squeeze this time?

In [ ]:
COMPACT_AT = 0.8        # trigger threshold: compact at 80% of the budget, not at 100%
KEEP_RECENT = 4         # the newest messages are never summarised away

COMPACTION_PROMPT = (
    "You compress the transcript of a research agent so the run can continue. Write a "
    "compact bullet summary that MUST preserve: decisions made and their reasons, every "
    "constraint stated by the user, verdicts on sources (id + judgement), open questions, "
    "and file names. You MAY drop: raw page text, tool dumps, superseded drafts — anything "
    "re-fetchable with one tool call. Output only the summary."
)


def compact(messages, budget=BUDGET):
    """Replace the oldest unpinned messages with a model-written summary.
    Returns (messages, did_compact)."""
    if context_tokens(messages) < ___ * budget:
        return messages, False                        # still enough headroom
    head, old, recent = messages[:2], messages[2:-___], messages[-KEEP_RECENT:]
    if len(old) < 2:
        return messages, False                        # nothing worth summarising
    resp = ollama.chat(model=MODEL,
                       messages=[{"role": "system", "content": COMPACTION_PROMPT},
                                 {"role": "user", "content": json.dumps(old, default=str)}],
                       options={"temperature": 0.0})
    summary = {"role": "system", "content": "[Compacted history — summary of earlier steps]\n"
                                            + resp["message"]["content"]}
    return head + [___] + recent, True


if OLLAMA_OK:
    compacted, did = compact(transcript)
    print(f"compacted: {len(compacted)} messages, ~{context_tokens(compacted):,} tokens "
          f"(compaction fired: {did})")
    print("constraint 'European' survives:", mentions(compacted, "european"))
    print("\n--- the summary the model wrote ---\n")
    print(compacted[2]["content"][:900])
else:
    print("Ollama not reachable — compaction needs a model call; see Part A.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
COMPACT_AT = 0.8        # trigger threshold: compact at 80% of the budget, not at 100%
KEEP_RECENT = 4         # the newest messages are never summarised away

COMPACTION_PROMPT = (
    "You compress the transcript of a research agent so the run can continue. Write a "
    "compact bullet summary that MUST preserve: decisions made and their reasons, every "
    "constraint stated by the user, verdicts on sources (id + judgement), open questions, "
    "and file names. You MAY drop: raw page text, tool dumps, superseded drafts — anything "
    "re-fetchable with one tool call. Output only the summary."
)


def compact(messages, budget=BUDGET):
    """Replace the oldest unpinned messages with a model-written summary.
    Returns (messages, did_compact)."""
    if context_tokens(messages) < COMPACT_AT * budget:
        return messages, False                        # still enough headroom
    head, old, recent = messages[:2], messages[2:-KEEP_RECENT], messages[-KEEP_RECENT:]
    if len(old) < 2:
        return messages, False                        # nothing worth summarising
    resp = ollama.chat(model=MODEL,
                       messages=[{"role": "system", "content": COMPACTION_PROMPT},
                                 {"role": "user", "content": json.dumps(old, default=str)}],
                       options={"temperature": 0.0})
    summary = {"role": "system", "content": "[Compacted history — summary of earlier steps]\n"
                                            + resp["message"]["content"]}
    return head + [summary] + recent, True


if OLLAMA_OK:
    compacted, did = compact(transcript)
    print(f"compacted: {len(compacted)} messages, ~{context_tokens(compacted):,} tokens "
          f"(compaction fired: {did})")
    print("constraint 'European' survives:", mentions(compacted, "european"))
    print("\n--- the summary the model wrote ---\n")
    print(compacted[2]["content"][:900])
else:
    print("Ollama not reachable — compaction needs a model call; see Part A.")
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- The trigger compares against `COMPACT_AT * budget`, not `budget`: the summarisation call
  must hold the old history *and* produce the summary inside the same context — triggering at
  the hard limit fails exactly when the mechanism is most needed.
- The compaction prompt is the engineering: it names what MUST survive and what MAY die. The
  asymmetry justifying the drop list: re-fetchable detail costs one tool call to recover; a
  lost constraint silently corrupts the rest of the run.
- The summary is spliced in as a `system`-role message right after the pinned head, so it
  carries authority — and note the honest caveat: whatever the summary omits or misstates is
  gone irreversibly, and errors compound. Every compaction is a bet.
- If your model's summary *drops* the European-sources constraint: congratulations, you have
  reproduced the lecture's warning empirically. Tighten the prompt and try again (Part G).

</details>

> **Q:** Give the keep/drop heuristic for an engineered compaction prompt, and the asymmetry that justifies it.
<details><summary>Click for answer</summary>

Keep: decisions and their rationale, user-stated constraints, open questions, identifiers
like file paths, current error states. Drop: raw tool dumps, full retrieved documents,
superseded drafts. The asymmetry: dropped detail that is re-fetchable costs one tool call to
recover; a lost decision or constraint silently corrupts the rest of the run and may be
unrecoverable.
</details>

## Part E — The memory file: persistence in 14 lines

Now the state that **survives the run**. This is the lecture's code slide, verbatim in
spirit: a curated plain-text file, read whole, **append-only** with a date stamp, and — the
consequential choice — *the model decides what is worth a note*. The write policy travels in
the tool description (and, in Part F, in the system prompt).

Why plain text: human-readable (you can audit what the agent believes), versionable (git
gives history, diffs and rollback), portable (no schema, no service). Only `memory_append`
becomes a *tool*; reading happens once at session start — the hybrid architecture Part F
wires up.

In [ ]:
MEMORY = Path("memory/notes.md")
MEMORY.parent.mkdir(exist_ok=True)


def memory_read() -> str:
    """Return the agent's notes file (read whole — it is meant to stay small)."""
    if not MEMORY.exists():
        return "(no notes yet)"
    return MEMORY.read_text(encoding="utf-8") or "(no notes yet)"


def memory_reset():
    """Wipe the notes (used to give each experiment condition a clean slate)."""
    MEMORY.write_text("", encoding="utf-8")


def memory_append(entry: str) -> str:
    """Add one short, dated note (append-only: the file is an auditable ledger)."""
    line = f"- {date.today()}: {___}\n"
    with MEMORY.open(___, encoding="utf-8") as f:
        f.write(line)
    return "stored"


MEMORY_SCHEMAS = [
    {"type": "function", "function": {
        "name": "memory_append",
        "description": ("Store ONE short note that should survive this session: stable user "
                        "preferences, verified source verdicts, standing rules. Distill "
                        "first — store the conclusion, never raw page text or transient "
                        "task detail."),
        "parameters": {"type": "object",
                       "properties": {"entry": {"type": "string",
                                                "description": "one short, distilled note"}},
                       "required": [___]}}},
]

# smoke test — works without any LLM
memory_reset()                                       # this lab starts from a clean slate
memory_append("smoke test: the memory file is writable")
print(memory_read())
memory_reset()                                       # remove the smoke-test note again
print("smoke-test note removed; notes.md will be filled by the agent in Part F")

<details>
<summary><b>Click here for the solution</b></summary>

```python
MEMORY = Path("memory/notes.md")
MEMORY.parent.mkdir(exist_ok=True)


def memory_read() -> str:
    """Return the agent's notes file (read whole — it is meant to stay small)."""
    if not MEMORY.exists():
        return "(no notes yet)"
    return MEMORY.read_text(encoding="utf-8") or "(no notes yet)"


def memory_reset():
    """Wipe the notes (used to give each experiment condition a clean slate)."""
    MEMORY.write_text("", encoding="utf-8")


def memory_append(entry: str) -> str:
    """Add one short, dated note (append-only: the file is an auditable ledger)."""
    line = f"- {date.today()}: {entry}\n"
    with MEMORY.open("a", encoding="utf-8") as f:
        f.write(line)
    return "stored"


MEMORY_SCHEMAS = [
    {"type": "function", "function": {
        "name": "memory_append",
        "description": ("Store ONE short note that should survive this session: stable user "
                        "preferences, verified source verdicts, standing rules. Distill "
                        "first — store the conclusion, never raw page text or transient "
                        "task detail."),
        "parameters": {"type": "object",
                       "properties": {"entry": {"type": "string",
                                                "description": "one short, distilled note"}},
                       "required": ["entry"]}}},
]

# smoke test — works without any LLM
memory_reset()                                       # this lab starts from a clean slate
memory_append("smoke test: the memory file is writable")
print(memory_read())
memory_reset()                                       # remove the smoke-test note again
print("smoke-test note removed; notes.md will be filled by the agent in Part F")
```

</details>

> **Q:** State the three criteria for a fact being worth writing to long-term memory, with an example that satisfies all three.
<details><summary>Click for answer</summary>

Stable (still true next week), reusable (future runs will plausibly need it), and expensive
to re-derive (rediscovery costs searches, tokens, or user patience). Example: "the user
requires European primary sources and APA citations" — stable across runs, relevant to every
report, and otherwise re-elicited from the user each time. The panel's cost asymmetry
justifies the discipline: a useless memory is not free — it pollutes every future retrieval
and every future prompt.
</details>

## Part F — The two-session experiment: measure what carries over

The lecture's lab assignment, verbatim: *run the agent twice on related topics and measure —
not guess — what usefully carries over, and what carries over that should not have.*

**The scenario** (from the lecture's own example): on **Monday** the user asks for a report
on *battery recycling* and states standing requirements — only European sources, APA
citations, distrust industry press releases. On **Wednesday** the user asks for a follow-up
on *lithium supply chains* — and does **not** restate the requirements. Each `run_session`
call is one process lifetime: its message list (working memory) dies at return; only
`reports/` and `memory/notes.md` survive.

**Two conditions.** **A — amnesiac**: no memory file. **B — memory file**: the hybrid
architecture from the lecture's products slide — the notes file is *injected* into the
system prompt at session start (small curated core, automatic), while writes stay
*deliberate tool calls* (`memory_append`), plus an end-of-run **consolidation pass**. The
loop also manages its window with Part D's `compact()` — memory files and compaction
compose.

**Instrumentation** (the point of the exercise): per session we log searches issued, source
ids seen, compactions fired and billed input tokens; afterwards Part F.3 compares
Wednesday-A against Wednesday-B.

In [ ]:
TEMPERATURE = 0.2

RESEARCH_PROMPT = (
    "You are a careful research agent. Search the corpus with web_search (vary your "
    "queries, search at least twice), weigh the sources critically, then write a concise "
    "Markdown report — title, findings, and a Sources section citing the URLs you used — "
    "and save it with save_report. After the report is saved, answer with ONE plain "
    "sentence and no tool call."
)

WRITE_POLICY = (
    "\n\nYou have a persistent memory file. Use the memory_append tool for facts worth "
    "keeping across sessions: stable user preferences, verified source verdicts, standing "
    "rules. Store conclusions, never raw page text or transient task detail.\n\n"
    "## Your notes from earlier sessions\n"
)


def build_messages(task, memory_on):
    """Hybrid architecture (lecture, section 5): the curated core — the whole notes file —
    is INJECTED at session start; writes remain deliberate TOOL calls."""
    system = RESEARCH_PROMPT
    if memory_on:
        system += WRITE_POLICY + ___()
    return [{"role": "system", "content": system},
            {"role": "user", "content": task}]


def run_session(task, memory_on=False, label="run", budget=BUDGET, max_steps=12,
                verbose=True):
    """ONE research session = one process lifetime. Working memory (the message list)
    dies at return; only reports/ and memory/notes.md survive."""
    registry = dict(TOOLS)
    schemas = list(SCHEMAS)
    if memory_on:
        registry["memory_append"] = memory_append
        schemas = schemas + MEMORY_SCHEMAS
    messages = build_messages(task, memory_on)
    log = {"label": label, "memory_on": memory_on, "searches": [], "sources_seen": set(),
           "notes_written": 0, "compactions": 0, "billed_tokens": 0, "steps": 0}
    for _ in range(max_steps):
        log["billed_tokens"] += ___(messages)     # re-sending is re-paying
        resp = ollama.chat(model=MODEL, messages=messages, tools=schemas,
                           options={"temperature": TEMPERATURE})
        msg = to_dict(resp["message"])
        messages.append(msg)
        log["steps"] += 1
        if not msg.get(___):
            if verbose:
                print(f"[{label}] FINAL: {str(msg['content'])[:110]}")
            break
        for call in msg["tool_calls"]:
            name = call["function"]["name"]
            args = call["function"].get("arguments", {})
            if isinstance(args, str):
                args = json.loads(args)
            if name == "web_search":
                log["searches"].append(args.get("query", ""))
            if name == "memory_append":
                log["notes_written"] += 1
            obs = run_tool(call, registry)
            log["sources_seen"] |= set(re.findall(r"\[(\w+)\]", obs["content"]))
            messages.append(obs)
            if verbose:
                print(f"[{label}] step {log['steps']:>2}: {name}")
        messages, did = compact(___, budget)
        log["compactions"] += int(did)
    log["sources_seen"] = sorted(log["sources_seen"] & ALL_IDS)
    log["transcript"] = messages
    return log

<details>
<summary><b>Click here for the solution</b></summary>

```python
TEMPERATURE = 0.2

RESEARCH_PROMPT = (
    "You are a careful research agent. Search the corpus with web_search (vary your "
    "queries, search at least twice), weigh the sources critically, then write a concise "
    "Markdown report — title, findings, and a Sources section citing the URLs you used — "
    "and save it with save_report. After the report is saved, answer with ONE plain "
    "sentence and no tool call."
)

WRITE_POLICY = (
    "\n\nYou have a persistent memory file. Use the memory_append tool for facts worth "
    "keeping across sessions: stable user preferences, verified source verdicts, standing "
    "rules. Store conclusions, never raw page text or transient task detail.\n\n"
    "## Your notes from earlier sessions\n"
)


def build_messages(task, memory_on):
    """Hybrid architecture (lecture, section 5): the curated core — the whole notes file —
    is INJECTED at session start; writes remain deliberate TOOL calls."""
    system = RESEARCH_PROMPT
    if memory_on:
        system += WRITE_POLICY + memory_read()
    return [{"role": "system", "content": system},
            {"role": "user", "content": task}]


def run_session(task, memory_on=False, label="run", budget=BUDGET, max_steps=12,
                verbose=True):
    """ONE research session = one process lifetime. Working memory (the message list)
    dies at return; only reports/ and memory/notes.md survive."""
    registry = dict(TOOLS)
    schemas = list(SCHEMAS)
    if memory_on:
        registry["memory_append"] = memory_append
        schemas = schemas + MEMORY_SCHEMAS
    messages = build_messages(task, memory_on)
    log = {"label": label, "memory_on": memory_on, "searches": [], "sources_seen": set(),
           "notes_written": 0, "compactions": 0, "billed_tokens": 0, "steps": 0}
    for _ in range(max_steps):
        log["billed_tokens"] += context_tokens(messages)     # re-sending is re-paying
        resp = ollama.chat(model=MODEL, messages=messages, tools=schemas,
                           options={"temperature": TEMPERATURE})
        msg = to_dict(resp["message"])
        messages.append(msg)
        log["steps"] += 1
        if not msg.get("tool_calls"):
            if verbose:
                print(f"[{label}] FINAL: {str(msg['content'])[:110]}")
            break
        for call in msg["tool_calls"]:
            name = call["function"]["name"]
            args = call["function"].get("arguments", {})
            if isinstance(args, str):
                args = json.loads(args)
            if name == "web_search":
                log["searches"].append(args.get("query", ""))
            if name == "memory_append":
                log["notes_written"] += 1
            obs = run_tool(call, registry)
            log["sources_seen"] |= set(re.findall(r"\[(\w+)\]", obs["content"]))
            messages.append(obs)
            if verbose:
                print(f"[{label}] step {log['steps']:>2}: {name}")
        messages, did = compact(messages, budget)
        log["compactions"] += int(did)
    log["sources_seen"] = sorted(log["sources_seen"] & ALL_IDS)
    log["transcript"] = messages
    return log
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `build_messages` is where the architecture fork is decided: the notes land in the *system
  prompt* (automatic injection — reliable, but silent influence with maximal authority; keep
  that in mind for R3), while `memory_append` stays in the tool list (deliberate, auditable,
  visible in the log as `notes_written`).
- The billing line adds the *full* current context before every call — that is the honest
  cost model from Part B: working memory is a subscription, not a one-off purchase.
- `compact()` runs inside the loop, after each step's observations: within-run window
  management and cross-run memory operate simultaneously, exactly the composition the
  lecture described (compaction manages the window, notes keep the knowledge).
- `sources_seen` is harvested from the `[id]` markers in tool observations and intersected
  with `ALL_IDS` — cheap, deterministic instrumentation that Part F.3 turns into metrics.

</details>

In [ ]:
CONSOLIDATION_PROMPT = (
    "Review this research-session transcript. Write AT MOST three short notes worth "
    "keeping for future sessions: stable user preferences, verified source verdicts, "
    "standing rules. One note per line, each line starting with '* '. Store conclusions, "
    "never raw page text or one-off task detail. If nothing qualifies, answer NONE."
)


def consolidate(log):
    """End-of-run reflection pass (lecture: 'mature systems do both'): review the
    transcript, distill at most three notes, append them to the memory file."""
    resp = ollama.chat(model=MODEL,
                       messages=[{"role": "system", "content": CONSOLIDATION_PROMPT},
                                 {"role": "user",
                                  "content": json.dumps(log["transcript"], default=str)}],
                       options={"temperature": 0.0})
    notes = [ln.strip().lstrip("*").strip() for ln in resp["message"]["content"].splitlines()
             if ln.strip().startswith(___)]
    for n in notes[:___]:
        memory_append(n)
    return notes


def report_text(filename):
    p = Path("reports") / filename
    return p.read_text(encoding="utf-8").lower() if p.exists() else ""


TASK_MON = ("Monday. Write a short report on battery recycling in Europe and save it as "
            "'battery_recycling.md'. My standing requirements for ALL reports I ask you "
            "for: use only European sources, cite in APA style, and treat industry press "
            "releases as unreliable.")
TASK_WED = ("Wednesday. Follow-up task: write a short report on lithium supply chains and "
            "save it as 'lithium_supply.md'.")        # requirements deliberately NOT restated

RUNS = {}
if OLLAMA_OK:
    print("=== Condition A — amnesiac agent (no memory file) ===")
    RUNS["A_mon"] = run_session(TASK_MON, memory_on=False, label="A_mon")
    RUNS["A_wed"] = run_session(TASK_WED, memory_on=False, label="A_wed")
    RUNS["A_wed"]["report_text"] = report_text("lithium_supply.md")

    print("\n=== Condition B — memory file (hybrid: inject at start, tool to write) ===")
    memory_reset()                                    # fresh memory for condition B
    RUNS["B_mon"] = run_session(TASK_MON, memory_on=___, label="B_mon")
    print("consolidated after Monday:", consolidate(RUNS["B_mon"]))
    RUNS["B_wed"] = run_session(TASK_WED, memory_on=True, label="B_wed")
    RUNS["B_wed"]["report_text"] = report_text("lithium_supply.md")
    print("consolidated after Wednesday:", consolidate(RUNS["B_wed"]))

    print("\n--- memory/notes.md after both sessions ---")
    print(memory_read())
else:
    print("Ollama not reachable — the experiment needs live runs; see Part A.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
CONSOLIDATION_PROMPT = (
    "Review this research-session transcript. Write AT MOST three short notes worth "
    "keeping for future sessions: stable user preferences, verified source verdicts, "
    "standing rules. One note per line, each line starting with '* '. Store conclusions, "
    "never raw page text or one-off task detail. If nothing qualifies, answer NONE."
)


def consolidate(log):
    """End-of-run reflection pass (lecture: 'mature systems do both'): review the
    transcript, distill at most three notes, append them to the memory file."""
    resp = ollama.chat(model=MODEL,
                       messages=[{"role": "system", "content": CONSOLIDATION_PROMPT},
                                 {"role": "user",
                                  "content": json.dumps(log["transcript"], default=str)}],
                       options={"temperature": 0.0})
    notes = [ln.strip().lstrip("*").strip() for ln in resp["message"]["content"].splitlines()
             if ln.strip().startswith("*")]
    for n in notes[:3]:
        memory_append(n)
    return notes


def report_text(filename):
    p = Path("reports") / filename
    return p.read_text(encoding="utf-8").lower() if p.exists() else ""


TASK_MON = ("Monday. Write a short report on battery recycling in Europe and save it as "
            "'battery_recycling.md'. My standing requirements for ALL reports I ask you "
            "for: use only European sources, cite in APA style, and treat industry press "
            "releases as unreliable.")
TASK_WED = ("Wednesday. Follow-up task: write a short report on lithium supply chains and "
            "save it as 'lithium_supply.md'.")        # requirements deliberately NOT restated

RUNS = {}
if OLLAMA_OK:
    print("=== Condition A — amnesiac agent (no memory file) ===")
    RUNS["A_mon"] = run_session(TASK_MON, memory_on=False, label="A_mon")
    RUNS["A_wed"] = run_session(TASK_WED, memory_on=False, label="A_wed")
    RUNS["A_wed"]["report_text"] = report_text("lithium_supply.md")

    print("\n=== Condition B — memory file (hybrid: inject at start, tool to write) ===")
    memory_reset()                                    # fresh memory for condition B
    RUNS["B_mon"] = run_session(TASK_MON, memory_on=True, label="B_mon")
    print("consolidated after Monday:", consolidate(RUNS["B_mon"]))
    RUNS["B_wed"] = run_session(TASK_WED, memory_on=True, label="B_wed")
    RUNS["B_wed"]["report_text"] = report_text("lithium_supply.md")
    print("consolidated after Wednesday:", consolidate(RUNS["B_wed"]))

    print("\n--- memory/notes.md after both sessions ---")
    print(memory_read())
else:
    print("Ollama not reachable — the experiment needs live runs; see Part A.")
```

</details>

In [ ]:
EUROPEAN = {d["id"] for d in corpus if d["region"] == ___}
UNRELIABLE = {d["id"] for d in corpus
              if d["source_type"] in ("industry press release", "marketing blog",
                                      "vendor pitch")}


def cited(report, doc_id):
    """Is a corpus document recognisably cited in a saved report?"""
    doc = next(d for d in corpus if d["id"] == doc_id)
    return (doc_id in report or doc["url"].lower() in report
            or doc["title"].lower()[:30] in report)


def wednesday_metrics(run_mon, run_wed):
    """What carried over from Monday to Wednesday — and what should not have."""
    rep = run_wed.get("report_text", "")
    return {
        "searches issued (Wed)": len(run_wed["searches"]),
        "Monday sources re-fetched (Wed)": len(set(run_mon["sources_seen"])
                                               ___ set(run_wed["sources_seen"])),
        "non-European sources cited": sum(cited(rep, i) for i in ALL_IDS - EUROPEAN),
        "unreliable sources cited": sum(cited(rep, i) for i in ___),
        "report mentions APA": int("apa" in rep),
        "compactions (Wed)": run_wed["compactions"],
        "billed input tokens (Wed)": run_wed["billed_tokens"],
    }


if OLLAMA_OK and RUNS:
    dfm = pd.DataFrame({"A (no memory)": wednesday_metrics(RUNS["A_mon"], RUNS["A_wed"]),
                        "B (memory file)": wednesday_metrics(RUNS["B_mon"], RUNS["B_wed"])})
    print("Wednesday run, compared across conditions:\n")
    print(dfm.to_string())
    print("\nRead the table against the lecture: preference carry-over should show up as "
          "fewer\nnon-European / unreliable citations and an APA mention in condition B; "
          "verdict\ncarry-over as fewer redundant fetches. Also check the failure side "
          "(R1, R3):\nwhat is in notes.md that should NOT be there?")
else:
    print("Ollama not reachable — metrics need the live runs from Part F.2.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
EUROPEAN = {d["id"] for d in corpus if d["region"] == "EU"}
UNRELIABLE = {d["id"] for d in corpus
              if d["source_type"] in ("industry press release", "marketing blog",
                                      "vendor pitch")}


def cited(report, doc_id):
    """Is a corpus document recognisably cited in a saved report?"""
    doc = next(d for d in corpus if d["id"] == doc_id)
    return (doc_id in report or doc["url"].lower() in report
            or doc["title"].lower()[:30] in report)


def wednesday_metrics(run_mon, run_wed):
    """What carried over from Monday to Wednesday — and what should not have."""
    rep = run_wed.get("report_text", "")
    return {
        "searches issued (Wed)": len(run_wed["searches"]),
        "Monday sources re-fetched (Wed)": len(set(run_mon["sources_seen"])
                                               & set(run_wed["sources_seen"])),
        "non-European sources cited": sum(cited(rep, i) for i in ALL_IDS - EUROPEAN),
        "unreliable sources cited": sum(cited(rep, i) for i in UNRELIABLE),
        "report mentions APA": int("apa" in rep),
        "compactions (Wed)": run_wed["compactions"],
        "billed input tokens (Wed)": run_wed["billed_tokens"],
    }


if OLLAMA_OK and RUNS:
    dfm = pd.DataFrame({"A (no memory)": wednesday_metrics(RUNS["A_mon"], RUNS["A_wed"]),
                        "B (memory file)": wednesday_metrics(RUNS["B_mon"], RUNS["B_wed"])})
    print("Wednesday run, compared across conditions:\n")
    print(dfm.to_string())
    print("\nRead the table against the lecture: preference carry-over should show up as "
          "fewer\nnon-European / unreliable citations and an APA mention in condition B; "
          "verdict\ncarry-over as fewer redundant fetches. Also check the failure side "
          "(R1, R3):\nwhat is in notes.md that should NOT be there?")
else:
    print("Ollama not reachable — metrics need the live runs from Part F.2.")
```

</details>

> **Q:** What is the fundamental weakness of *pure* tool-based memory access — and how does the hybrid pattern used in this lab address it?
<details><summary>Click for answer</summary>

The model must notice that it lacks knowledge before it will query, but nothing in the
current context indicates that a relevant memory exists — the agent does not know what it
has forgotten, so relevant memories go unconsulted. The hybrid (this lab's `build_messages`)
fixes reliability for the few always-relevant facts by injecting the small curated core at
session start, while keeping writes (and, in larger systems, long-tail retrieval) behind
explicit, auditable tool calls. The price of the injected half: tokens on every call and
silent influence — the poisoning surface of R3.
</details>

> **📝 Report task R1:** After the Part F experiment, open `memory/notes.md`. Classify **every** entry the agent stored using the lecture's CoALA taxonomy (episodic / semantic / procedural — short-term is by definition absent from a file that survived the run). Then audit the write policy: name at least one entry that **usefully carried over** to Wednesday and at least one that **should not have been stored** (or argue why none exists), judging each against the three write criteria *stable / reusable / expensive to re-derive*.
> *No solution is provided — include your answer and a short justification in your lab report.*

> **Q:** Give the four forgetting mechanisms from the lecture and match each to a failure it prevents.
<details><summary>Click for answer</summary>

Time-based expiry (TTL) prevents predictably volatile facts — sprint goals, deadlines — from
outliving their truth. Usage-based decay prevents never-consulted entries from accumulating
and diluting retrieval. Supersession prevents contradictory beliefs from coexisting after a
fact changes ("prefers React" replaces "prefers Vue"). Explicit deletion serves user control
and legal erasure obligations (GDPR Art. 17). A store without deletion degrades into noise.
</details>

> **📝 Report task R2 (code):** Complete the cell below — the **forgetting pass** the lecture demanded ("pruning runs as a separate consolidation pass"): TTL expiry for volatile notes plus supersession, so that for each `(tag, subject)` only the newest note survives. Then answer in your report: which two of the lecture's **four forgetting mechanisms** does this pass implement, which two are missing, and why must pruning run as a *separate* pass rather than inside `memory_append` at write time?
> *No solution is provided — include your code and a short justification in your lab report.*

In [ ]:
# A synthetic memory file that has aged badly — run your forgetting pass on it.
# Note format:  - YYYY-MM-DD [tag] subject: text     (tags: pref, rule, task)
STALE = """\
- 2026-04-01 [rule] quotes: always verify against the original source before citing
- 2026-05-02 [pref] citations: APA style
- 2026-05-02 [task] battery report: draft v1 saved, sources still unverified
- 2026-05-20 [pref] sources: only European sources
- 2026-05-21 [pref] sources: only European sources
- 2026-06-10 [pref] citations: IEEE style
- 2026-06-28 [task] lithium report: open question on recycling-rate data
"""
STALE_PATH = Path("memory/stale_notes.md")
STALE_PATH.parent.mkdir(exist_ok=True)
STALE_PATH.write_text(STALE, encoding="utf-8")

TTL_DAYS = {"task": 14, "pref": None, "rule": None}     # None = never expires
NOTE = re.compile(r"- (\d{4}-\d{2}-\d{2}) \[(\w+)\] ([^:]+): (.+)")


def consolidate_memory(path, today):
    """The forgetting pass (run BETWEEN runs, never during one): TTL expiry for volatile
    notes, then supersession — per (tag, subject) key only the NEWEST note survives.
    Rewrites the file in place and returns the surviving lines."""
    kept = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        m = NOTE.match(line)
        if not m:
            continue
        day, tag, subject, text = m.groups()
        age = (today - datetime.strptime(day, "%Y-%m-%d").date()).days
        ttl = TTL_DAYS.get(tag)
        if ttl is not None and age > ___:
            continue                                    # expired: forget it
        key = (___, subject.strip().lower())
        if key not in kept or day > kept[key][___]:       # supersession: newest wins
            kept[key] = (day, line)
    lines = sorted(line for day, line in kept.values())
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return lines


print("surviving notes (expected: rule kept, ONE sources note, citations superseded "
      "to IEEE,\nold task expired, recent task kept):\n")
for line in consolidate_memory(STALE_PATH, today=date(2026, 7, 2)):
    print(line)

> **📝 Report task R3:** The corpus page `supplier_pitch` contains a planted **memory-poisoning** attempt (read its text in `data/corpus.json`). Check your condition-B artefacts: did the instruction reach `memory/notes.md` or the Wednesday report? Whatever you observed, explain in your report (a) the mechanism by which memory upgrades a *transient* indirect prompt injection (Greshake et al., 2023) into a *persistent* compromise, and (b) the **two architectural defenses** from the lecture, stating concretely how each would have to change this lab's `memory_append` / consolidation pipeline.
> *No solution is provided — include your answer and a short justification in your lab report.*

## Part G — Tuning & exploration

No gaps here — play. All knobs of the memory system in one place; each maps to a design
decision from the lecture:

- **`BUDGET`** — the token budget. Shrink it to 1,600 and compaction fires on almost every
  step: watch billed tokens fall and summary errors compound. Raise it to 6,000 and
  compaction may never fire.
- **`COMPACT_AT`** — push the trigger towards 0.95 and compaction runs out of headroom
  exactly as the lecture predicted; drop it to 0.5 and you pay for summaries you did not
  need yet.
- **`KEEP_RECENT`** — how much verbatim train-of-thought survives each squeeze.
- **`TEMPERATURE`** — determinism vs variety across repeated experiment runs.
- **Ablation ideas:** rerun Part F with `WRITE_POLICY` stripped from the system prompt (is
  `memory_append` ever called on its own? — the tool-only weakness, live); or replace
  `compact` with `truncate` inside `run_session` and measure which constraints die; or edit
  `notes.md` by hand and plant a *stale* preference ("cite in IEEE style") to watch a
  confidently wrong memory beat no memory.

In [ ]:
# ---- tuning playground (no gaps) -------------------------------------------
def apply_tuning(budget=2_400, compact_at=0.8, keep_recent=4, temperature=0.2):
    """Set the global knobs used by compact() and run_session()."""
    global BUDGET, COMPACT_AT, KEEP_RECENT, TEMPERATURE
    BUDGET, COMPACT_AT, KEEP_RECENT, TEMPERATURE = (budget, compact_at, keep_recent,
                                                    temperature)
    print(f"applied: BUDGET={budget}, COMPACT_AT={compact_at}, "
          f"KEEP_RECENT={keep_recent}, TEMPERATURE={temperature}")


def quick_wednesday(budget=2_400, compact_at=0.8, keep_recent=4, temperature=0.2):
    """One Wednesday run with fresh settings, against the CURRENT memory file."""
    apply_tuning(budget, compact_at, keep_recent, temperature)
    if not OLLAMA_OK:
        print("Ollama not reachable — tuning needs live runs; see Part A.")
        return
    log = run_session(TASK_WED, memory_on=True, label=f"tune_b{budget}", budget=budget,
                      verbose=False)
    print(f"→ steps={log['steps']}, searches={len(log['searches'])}, "
          f"compactions={log['compactions']}, notes written={log['notes_written']}, "
          f"billed≈{log['billed_tokens']:,} tokens")


try:
    import ipywidgets as widgets
    from IPython.display import display
    display(widgets.interactive(
        quick_wednesday,
        budget=widgets.IntSlider(min=1_200, max=6_000, step=400, value=2_400),
        compact_at=widgets.FloatSlider(min=0.5, max=0.95, step=0.05, value=0.8),
        keep_recent=widgets.IntSlider(min=2, max=10, step=1, value=4),
        temperature=widgets.FloatSlider(min=0.0, max=1.0, step=0.1, value=0.2)))
except Exception:
    print("ipywidgets not available — call quick_wednesday(...) manually, e.g.:")
    print("  quick_wednesday(budget=1600, compact_at=0.7)")
    print("  quick_wednesday(budget=6000)   # compaction may never fire")

## Wrap-up

**Takeaways**

- The context window is **working memory**: volatile, re-billed on every call (your Part B
  meter made the 4-million-token bigfact concrete), and bounded — the word *memory* is
  reserved for state that survives the run.
- Within a run you managed the window three ways and **measured** the differences:
  truncation with pinning lost the mid-run constraint *silently*; engineered compaction
  (80% trigger, explicit keep/drop) preserved it in a fraction of the tokens — as a lossy,
  irreversible bet; the memory file survived both compaction and process exit.
- Across runs, **policies did the real work**: a write policy in the tool description and
  system prompt (stable / reusable / expensive to re-derive, distill first), the hybrid read
  architecture (inject the curated core, keep writes as auditable tool calls), and
  forgetting as a separate consolidation pass (R2: TTL + supersession).
- The two-session experiment showed memory as a **measured** capability: preference and
  verdict carry-over in condition B — and as a **liability surface**: whatever `notes.md`
  now contains that should not be there (R1), including, possibly, a supplier's planted
  instruction (R3).

**Next week:** Session 09 — *Vector databases and retrieval-augmented generation*: the heavy
machinery for semantic memory at scale, when the notes no longer fit in any window and
retrieval must find the right needle.

---

### 📝 For your lab report

| Task | What to hand in |
|---|---|
| **R1** | The memory-file audit: every entry classified (episodic / semantic / procedural), one useful carry-over and one write-policy violation, each judged against *stable / reusable / expensive to re-derive* |
| **R2** | Your completed forgetting pass **plus**: which two of the four forgetting mechanisms it implements, which two are missing, and why pruning must be a separate pass |
| **R3** | The memory-poisoning analysis: what you observed in `notes.md` / the Wednesday report, the transient-to-persistent mechanism, and the two architectural defenses applied concretely to this lab's pipeline |

*Reminder: report tasks have no solutions in this notebook — your own reasoning is the deliverable.*